In [0]:
SELECT
  b.account_name,
  b.account_executive,
  b.usecase_id,
  concat('<a href="https://databricks.lightning.force.com/lightning/r/UseCase__c/', b.usecase_id, '/view" target="_blank">', b.use_case_name, '</a>') AS usecase_url,
  u.estimated_monthly_dollar_dbus,
  b.blocker_id,
  b.blocker_name,
  b.category,
  b.type,
  case when b.type = 'Blocked' then u.estimated_monthly_dollar_dbus else 0 end as blocked_dbus,
  case when b.type = 'Blocked' then 1 else 0 end as blocked_count,
  case when b.type = 'Friction' then u.estimated_monthly_dollar_dbus else 0 end as friction_dbus,
  case when b.type = 'Friction' then 1 else 0 end as friction_count,
  b.comment,
  aha_item.aha_reference,
  CASE
    WHEN aha_item.aha_reference IS NOT NULL
    THEN '<a href="https://databrickinternal.ideas.aha.io/ideas/' || aha_item.aha_reference || '" target="_blank">' || aha_item.aha_reference || '</a>'
  END                       AS aha_reference_url,
  aha_item.aha_name         AS aha_idea,
  aha_item.aha_status       AS aha_status,
  aha_item.aha_link         AS aha_link
FROM main.gtm_silver.blocker_detail b
INNER JOIN main.gtm_silver.use_case_detail u
  ON b.usecase_id = u.usecase_id
LATERAL VIEW OUTER EXPLODE(b.aha) AS aha_item
WHERE b.region_level_3 = 'Italy Strategic Core'
  AND b.snapshot_date  = current_date()
  AND (:ae_email = '' OR b.concatenated_emails LIKE '%' || :ae_email || '%')
  AND u.stage_number BETWEEN 1 AND 5
ORDER BY
  b.account_name,
  b.use_case_name,
  b.blocker_name